In [4]:
import pandas as pd

s1 = pd.read_csv("train_source1.tsv", sep="\t")
s2 = pd.read_csv("train_source2.tsv", sep="\t")
s3 = pd.read_csv("train_source3.tsv", sep="\t")
gt = pd.read_csv("train_ground_truth.tsv", sep="\t")

print("Source 1 shape:", s1.shape)
print("Source 2 shape:", s2.shape)
print("Source 3 shape:", s3.shape)
print("Ground Truth shape:", gt.shape)

Source 1 shape: (2206821, 4)
Source 2 shape: (5034616, 4)
Source 3 shape: (5285603, 4)
Ground Truth shape: (2206821, 2)


In [5]:
print("\nSource 1:")
display(s1.head())

print("\nSource 2:")
display(s2.head())

print("\nSource 3:")
display(s3.head())

print("\nGround Truth:")
display(gt.head())


Source 1:


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India



Source 2:


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



Source 3:


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India



Ground Truth:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [6]:
# ==========================================
# STEP 1: DATASET ANALYSIS
# ==========================================

print("========== DATASET SIZES ==========")
print("Source 1:", s1.shape)
print("Source 2:", s2.shape)
print("Source 3:", s3.shape)
print("Ground Truth:", gt.shape)


print("\n========== COLUMNS ==========")
print("Source 1:", list(s1.columns))
print("Source 2:", list(s2.columns))
print("Source 3:", list(s3.columns))
print("Ground Truth:", list(gt.columns))


print("\n========== MISSING VALUES ==========")

print("\nSource 1:")
print(s1.isnull().sum())

print("\nSource 2:")
print(s2.isnull().sum())

print("\nSource 3:")
print(s3.isnull().sum())


print("\n========== UNIQUE COUNTRIES ==========")

print("\nSource 1:")
print(s1["country"].value_counts(dropna=False))

print("\nSource 2:")
print(s2["country"].value_counts(dropna=False))

print("\nSource 3:")
print(s3["country"].value_counts(dropna=False))


print("\n========== GROUND TRUTH SAMPLE ==========")
print(gt.head(10).to_string(index=False))

========== DATASET SIZES ==========
Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)

========== COLUMNS ==========
Source 1: ['entity_id', 'business_name', 'business_address', 'country']
Source 2: ['entity_id', 'business_name', 'business_address', 'country']
Source 3: ['entity_id', 'business_name', 'business_address', 'country']
Ground Truth: ['source1_entity_id', 'matched_entity_ids']

========== MISSING VALUES ==========

Source 1:
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

Source 2:
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

Source 3:
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

========== UNIQUE COUNTRIES ==========

Source 1:
country
US       1323633
India     883188
Name: count, dtype: int64

Source 2:
country
US       30

In [7]:
# ==========================================
# STEP 2: MATCH COUNT ANALYSIS
# ==========================================

matched = gt["matched_entity_ids"].fillna("").astype(str)

# Empty = 0 matches
# One ID = 1 match
# Two IDs separated by comma = 2 matches, etc.
gt["match_count"] = (
    matched.ne("").astype(int) +
    matched.str.count(",")
)

print("========== MATCH COUNT DISTRIBUTION ==========")
print(gt["match_count"].value_counts().sort_index())

print("\n========== MATCH COUNT STATISTICS ==========")
print(gt["match_count"].describe())

print("\nNumber of entities with NO MATCH:")
print((gt["match_count"] == 0).sum())

print("\nNumber of entities with EXACTLY ONE MATCH:")
print((gt["match_count"] == 1).sum())

print("\nNumber of entities with MULTIPLE MATCHES:")
print((gt["match_count"] > 1).sum())

========== MATCH COUNT DISTRIBUTION ==========
match_count
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

========== MATCH COUNT STATISTICS ==========
count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: match_count, dtype: float64

Number of entities with NO MATCH:
123247

Number of entities with EXACTLY ONE MATCH:
119157

Number of entities with MULTIPLE MATCHES:
1964417


In [8]:
# ==========================================
# STEP 3: INSPECT REAL MATCHES
# ==========================================

for i in range(5):

    s1_id = gt.iloc[i]["source1_entity_id"]
    matched_ids = str(gt.iloc[i]["matched_entity_ids"])

    print("\n" + "=" * 100)
    print("SOURCE 1:", s1_id)
    print("MATCHES:", matched_ids)

    print("\n--- Source 1 Record ---")
    display(
        s1[s1["entity_id"] == s1_id]
    )

    if matched_ids != "nan" and matched_ids.strip() != "":

        ids = matched_ids.split(",")

        print("\n--- Matching Source 2 Records ---")
        display(
            s2[s2["entity_id"].isin(ids)]
        )

        print("\n--- Matching Source 3 Records ---")
        display(
            s3[s3["entity_id"].isin(ids)]
        )


SOURCE 1: S1-965667
MATCHES: S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364

--- Source 1 Record ---


,entity_id,business_name,business_address,country
1286323,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US



--- Matching Source 2 Records ---


,entity_id,business_name,business_address,country
1376739,S2-681193310,Maure Wilblims Colombier Inc,NaN,US
2003754,S2-743505751,Maure Williams Colombier,NaN,US



--- Matching Source 3 Records ---


...

,entity_id,business_name,business_address,country
539414,S1-102811957,Payne Enterprises,"3315 Fremont Street, Peoria, IL",US



--- Matching Source 2 Records ---


,entity_id,business_name,business_address,country
385243,S2-625774905,PAYNE-ENRTPRMISES,"3315 FREMONT SAINT, PEORIA, IL",US
731099,S2-478959098,Payne Énterprises,"3315 FREMONT ST, PEORIA, IL",US
1776076,S2-553508714,Payne Enterpires,"3315 FREMONT ST, PEORIA, IL",US



--- Matching Source 3 Records ---


,entity_id,business_name,business_address,country
666012,S3-449308785,Payne Enterprises LLC,"Fremont St, Peoria, Illinois",US
876298,S3-728090388,Payne Etrepndiels,"3315 Fremont St, Peoria, Illinois",US
4587222,S3-928796641,Payne Énterprises,"3315 Fremont Street, Peoria, Illinois",US


In [9]:
import re
import unicodedata
import pandas as pd
import numpy as np

In [10]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower().strip()
    
    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)
    
    # Replace punctuation with spaces
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def normalize_name(text):
    text = normalize_text(text)
    
    # Common business abbreviations
    replacements = {
        "pvt": "private",
        "ltd": "limited",
        "corp": "corporation",
        "co": "company",
        "inc": "incorporated"
    }
    
    words = text.split()
    words = [replacements.get(word, word) for word in words]
    
    return " ".join(words)


def normalize_address(text):
    text = normalize_text(text)
    
    replacements = {
        "rd": "road",
        "st": "street",
        "ave": "avenue",
        "av": "avenue",
        "blvd": "boulevard",
        "ln": "lane",
        "dr": "drive",
        "hwy": "highway"
    }
    
    words = text.split()
    words = [replacements.get(word, word) for word in words]
    
    return " ".join(words)

In [11]:
for df in [s1, s2, s3]:
    
    df["name_norm"] = df["business_name"].apply(normalize_name)
    df["address_norm"] = df["business_address"].apply(normalize_address)
    df["country_norm"] = df["country"].fillna("").astype(str).str.lower().str.strip()

In [12]:
display(
    s1[
        [
            "business_name",
            "name_norm",
            "business_address",
            "address_norm",
            "country"
        ]
    ].head(10)
)

,business_name,name_norm,business_address,address_norm,country
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US
2,B+ Retail Inc,b retail incorporated,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US
8,Moore Bitwise Inc,moore bitwise incorporated,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US


In [13]:
# Create lookup tables for exact normalized name + country

s2_name_blocks = (
    s2[
        ["entity_id", "name_norm", "country_norm"]
    ]
    .groupby(["country_norm", "name_norm"])["entity_id"]
    .agg(list)
    .to_dict()
)

s3_name_blocks = (
    s3[
        ["entity_id", "name_norm", "country_norm"]
    ]
    .groupby(["country_norm", "name_norm"])["entity_id"]
    .agg(list)
    .to_dict()
)

print("S2 name blocks:", len(s2_name_blocks))
print("S3 name blocks:", len(s3_name_blocks))

S2 name blocks: 3971628
S3 name blocks: 4223576


In [14]:
# ==========================================
# TEST NAME BLOCKING RECALL
# ==========================================

sample_gt = gt.sample(
    n=min(50000, len(gt)),
    random_state=42
).copy()

def get_name_candidates(row):
    key = (
        str(s1.loc[s1["entity_id"] == row["source1_entity_id"], "country_norm"].iloc[0]),
        str(s1.loc[s1["entity_id"] == row["source1_entity_id"], "name_norm"].iloc[0])
    )
    
    candidates = set()
    
    candidates.update(s2_name_blocks.get(key, []))
    candidates.update(s3_name_blocks.get(key, []))
    
    return candidates


# Build lookup for S1
s1_lookup = s1.set_index("entity_id")[["name_norm", "country_norm"]]

covered = 0
total_matches = 0

for _, row in sample_gt.iterrows():
    
    s1_id = row["source1_entity_id"]
    
    name = s1_lookup.loc[s1_id, "name_norm"]
    country = s1_lookup.loc[s1_id, "country_norm"]
    
    candidates = set()
    
    candidates.update(
        s2_name_blocks.get((country, name), [])
    )
    
    candidates.update(
        s3_name_blocks.get((country, name), [])
    )
    
    actual = set(
        str(row["matched_entity_ids"]).split(",")
    ) if str(row["matched_entity_ids"]).strip() else set()
    
    actual.discard("")
    actual.discard("nan")
    
    total_matches += len(actual)
    
    covered += len(actual.intersection(candidates))


print("Actual matches in sample:", total_matches)
print("Matches found by name blocking:", covered)

if total_matches > 0:
    print(
        "Candidate recall:",
        round(covered / total_matches, 4)
    )

Actual matches in sample: 172784
Matches found by name blocking: 42609
Candidate recall: 0.2466


In [15]:
from collections import defaultdict

def get_name_tokens(text):
    if not text:
        return []
    
    words = text.split()
    
    # Ignore very short tokens
    return [w for w in words if len(w) >= 3]


def build_token_index(df):
    index = defaultdict(list)
    
    for entity_id, name in zip(df["entity_id"], df["name_norm"]):
        tokens = get_name_tokens(name)
        
        for token in set(tokens):
            index[token].append(entity_id)
    
    return index


print("Building S2 token index...")
s2_token_index = build_token_index(s2)

print("Building S3 token index...")
s3_token_index = build_token_index(s3)

print("S2 token blocks:", len(s2_token_index))
print("S3 token blocks:", len(s3_token_index))

Building S2 token index...
Building S3 token index...
S2 token blocks: 808416
S3 token blocks: 860942


In [ ]:
# ==========================================
# TEST TOKEN BLOCKING RECALL
# ==========================================

covered = 0
total_matches = 0

s1_lookup = s1.set_index("entity_id")[["name_norm", "country_norm"]]

for _, row in sample_gt.iterrows():

    s1_id = row["source1_entity_id"]

    name = s1_lookup.loc[s1_id, "name_norm"]

    tokens = get_name_tokens(name)

    candidates = set()

    for token in tokens:
        candidates.update(s2_token_index.get(token, []))
        candidates.update(s3_token_index.get(token, []))

    actual = set(
        str(row["matched_entity_ids"]).split(",")
    )

    actual.discard("")
    actual.discard("nan")

    total_matches += len(actual)

    covered += len(actual.intersection(candidates))


print("Actual matches in sample:", total_matches)
print("Matches found by token blocking:", covered)

if total_matches > 0:
    print(
        "Token blocking recall:",
        round(covered / total_matches, 4)
    )

In [1]:
import os
import pandas as pd

In [2]:
def load_data_from_dir(directory):
    """
    Loads all TSV files from a directory into a dictionary of DataFrames.

    The first time:
        TSV -> DataFrame -> Parquet cache

    Next time:
        Parquet -> DataFrame
    """

    data = {}

    if not os.path.exists(directory):
        print(f"Warning: Directory '{directory}' not found.")
        return data

    for file in os.listdir(directory):

        # Only process TSV files
        if file.endswith(".tsv") and not file.startswith("._"):

            name = os.path.splitext(file)[0]

            # Path for cached Parquet file
            parquet_path = os.path.join(
                directory,
                f"{name}.parquet"
            )

            # If Parquet already exists, load it
            if os.path.exists(parquet_path):

                print(
                    f"Loading cached {name}.parquet...",
                    flush=True
                )

                df = pd.read_parquet(parquet_path)

            # Otherwise read TSV and create Parquet cache
            else:

                filepath = os.path.join(directory, file)

                print(
                    f"Reading {file} and caching to parquet...",
                    flush=True
                )

                df = pd.read_csv(
                    filepath,
                    sep="\t",
                    dtype=str
                )

                df.to_parquet(
                    parquet_path,
                    engine="pyarrow"
                )

            data[name] = df

    return data

In [3]:
DATA_DIR = r"C:\Users\anupr"

In [4]:
data = load_data_from_dir(DATA_DIR)

Reading train_ground_truth.tsv and caching to parquet...
Reading train_source1.tsv and caching to parquet...
Reading train_source2.tsv and caching to parquet...
Reading train_source3.tsv and caching to parquet...


In [5]:
s1 = data["train_source1"]
s2 = data["train_source2"]
s3 = data["train_source3"]
gt = data["train_ground_truth"]

In [6]:
print("Source 1:", s1.shape)
print("Source 2:", s2.shape)
print("Source 3:", s3.shape)
print("Ground Truth:", gt.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)


In [7]:
print("\nSource 1 columns:")
print(s1.columns.tolist())

print("\nSource 2 columns:")
print(s2.columns.tolist())

print("\nSource 3 columns:")
print(s3.columns.tolist())

print("\nGround Truth columns:")
print(gt.columns.tolist())


Source 1 columns:
['entity_id', 'business_name', 'business_address', 'country']

Source 2 columns:
['entity_id', 'business_name', 'business_address', 'country']

Source 3 columns:
['entity_id', 'business_name', 'business_address', 'country']

Ground Truth columns:
['source1_entity_id', 'matched_entity_ids']


In [8]:
print("\n========== SOURCE 1 ==========")
display(s1.head())

print("\n========== SOURCE 2 ==========")
display(s2.head())

print("\n========== SOURCE 3 ==========")
display(s3.head())

print("\n========== GROUND TRUTH ==========")
display(gt.head())


========== SOURCE 1 ==========


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India



========== SOURCE 2 ==========


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



========== SOURCE 3 ==========


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India



========== GROUND TRUTH ==========


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [9]:
import os
import pandas as pd

# ==========================================
# STEP 1: DATASET PATH
# ==========================================

TRAIN_DIR = r"C:\Users\anupr\Amazon ML Challenge\Train"

print("Training directory exists:", os.path.exists(TRAIN_DIR))
print("\nFiles in Train folder:")

for file in os.listdir(TRAIN_DIR):
    print(file)

Training directory exists: True

Files in Train folder:
train_ground_truth.parquet
train_ground_truth.tsv
train_source1.parquet
train_source1.tsv
train_source2.parquet
train_source2.tsv
train_source3.parquet
train_source3.tsv


In [10]:
# ==========================================
# STEP 2: LOAD TRAINING DATA
# ==========================================

print("Loading Source 1...")
s1 = pd.read_parquet(
    os.path.join(TRAIN_DIR, "train_source1.parquet")
)

print("Loading Source 2...")
s2 = pd.read_parquet(
    os.path.join(TRAIN_DIR, "train_source2.parquet")
)

print("Loading Source 3...")
s3 = pd.read_parquet(
    os.path.join(TRAIN_DIR, "train_source3.parquet")
)

print("Loading Ground Truth...")
gt = pd.read_parquet(
    os.path.join(TRAIN_DIR, "train_ground_truth.parquet")
)

print("\nAll datasets loaded successfully!")

Loading Source 1...
Loading Source 2...
Loading Source 3...
Loading Ground Truth...

All datasets loaded successfully!


In [11]:
print("\nDataset sizes:")
print("Source 1:", s1.shape)
print("Source 2:", s2.shape)
print("Source 3:", s3.shape)
print("Ground Truth:", gt.shape)


Dataset sizes:
Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)


In [12]:
# ==========================================
# STEP 3: VERIFY DATA
# ==========================================

print("========== COLUMNS ==========")

print("\nSource 1:")
print(s1.columns.tolist())

print("\nSource 2:")
print(s2.columns.tolist())

print("\nSource 3:")
print(s3.columns.tolist())

print("\nGround Truth:")
print(gt.columns.tolist())


print("\n========== DATA TYPES ==========")

print("\nSource 1:")
print(s1.dtypes)

print("\nSource 2:")
print(s2.dtypes)

print("\nSource 3:")
print(s3.dtypes)

print("\nGround Truth:")
print(gt.dtypes)

========== COLUMNS ==========

Source 1:
['entity_id', 'business_name', 'business_address', 'country']

Source 2:
['entity_id', 'business_name', 'business_address', 'country']

Source 3:
['entity_id', 'business_name', 'business_address', 'country']

Ground Truth:
['source1_entity_id', 'matched_entity_ids']

========== DATA TYPES ==========

Source 1:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object

Source 2:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object

Source 3:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object

Ground Truth:
source1_entity_id     object
matched_entity_ids    object
dtype: object


In [13]:
# ==========================================
# CHECK SAMPLE RECORDS
# ==========================================

print("\n========== SOURCE 1 SAMPLE ==========")
display(s1.head(3))

print("\n========== SOURCE 2 SAMPLE ==========")
display(s2.head(3))

print("\n========== SOURCE 3 SAMPLE ==========")
display(s3.head(3))

print("\n========== GROUND TRUTH SAMPLE ==========")
display(gt.head(3))


========== SOURCE 1 SAMPLE ==========


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US



========== SOURCE 2 SAMPLE ==========


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India



========== SOURCE 3 SAMPLE ==========


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,None,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US



========== GROUND TRUTH SAMPLE ==========


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"


In [14]:
# ==========================================
# STEP 4: TEXT NORMALIZATION
# ==========================================

import re
import unicodedata


def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).strip().lower()
    
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)
    
    # Replace punctuation with spaces
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def normalize_name(text):
    text = normalize_text(text)
    
    # Common business-name abbreviations
    replacements = {
        "pvt": "private",
        "ltd": "limited",
        "corp": "corporation",
        "co": "company",
        "inc": "incorporated"
    }
    
    words = text.split()
    words = [replacements.get(word, word) for word in words]
    
    return " ".join(words)


def normalize_address(text):
    text = normalize_text(text)
    
    # Common address abbreviations
    replacements = {
        "rd": "road",
        "st": "street",
        "ave": "avenue",
        "av": "avenue",
        "blvd": "boulevard",
        "ln": "lane",
        "dr": "drive",
        "hwy": "highway"
    }
    
    words = text.split()
    words = [replacements.get(word, word) for word in words]
    
    return " ".join(words)


print("Normalization functions created successfully.")

Normalization functions created successfully.


In [15]:
# ==========================================
# APPLY NORMALIZATION
# ==========================================

print("Normalizing Source 1...")
s1["name_norm"] = s1["business_name"].apply(normalize_name)
s1["address_norm"] = s1["business_address"].apply(normalize_address)
s1["country_norm"] = s1["country"].fillna("").astype(str).str.lower().str.strip()

print("Normalizing Source 2...")
s2["name_norm"] = s2["business_name"].apply(normalize_name)
s2["address_norm"] = s2["business_address"].apply(normalize_address)
s2["country_norm"] = s2["country"].fillna("").astype(str).str.lower().str.strip()

print("Normalizing Source 3...")
s3["name_norm"] = s3["business_name"].apply(normalize_name)
s3["address_norm"] = s3["business_address"].apply(normalize_address)
s3["country_norm"] = s3["country"].fillna("").astype(str).str.lower().str.strip()

print("\nNormalization completed.")

Normalizing Source 1...
Normalizing Source 2...
Normalizing Source 3...

Normalization completed.


In [16]:
display(
    s1[
        [
            "business_name",
            "name_norm",
            "business_address",
            "address_norm",
            "country",
            "country_norm"
        ]
    ].head(10)
)

,business_name,name_norm,business_address,address_norm,country,country_norm
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US,us
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US,us
2,B+ Retail Inc,b retail incorporated,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US,us
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US,us
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India,india
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US,us
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India,india
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US,us
8,Moore Bitwise Inc,moore bitwise incorporated,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US,us
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US,us


In [17]:
# ==========================================
# STEP 5: NAME TOKEN BLOCKING
# ==========================================

from collections import defaultdict


def get_name_tokens(name):
    """
    Split normalized business name into useful tokens.
    Very short tokens are ignored.
    """
    
    if not name:
        return []
    
    tokens = name.split()
    
    return [
        token
        for token in tokens
        if len(token) >= 3
    ]


def build_token_index(df):
    """
    Creates:
    
    token + country
        ↓
    list of entity IDs
    """
    
    index = defaultdict(list)
    
    for entity_id, name, country in zip(
        df["entity_id"],
        df["name_norm"],
        df["country_norm"]
    ):
        
        tokens = get_name_tokens(name)
        
        # Avoid adding the same entity twice
        # if a word appears more than once.
        for token in set(tokens):
            
            index[(country, token)].append(entity_id)
    
    return index


print("Building Source 2 token index...")
s2_token_index = build_token_index(s2)

print("Building Source 3 token index...")
s3_token_index = build_token_index(s3)

print("\nToken indexes created successfully!")

print("S2 token blocks:", len(s2_token_index))
print("S3 token blocks:", len(s3_token_index))

Building Source 2 token index...
Building Source 3 token index...

Token indexes created successfully!
S2 token blocks: 838931
S3 token blocks: 900475


In [21]:
# ==========================================
# STEP 6A: FAST RECALL TEST SETUP
# ==========================================

sample_gt = gt.sample(
    n=5000,
    random_state=42
).copy()

# Create fast entity lookups
s1_lookup = s1.set_index("entity_id")[
    ["name_norm", "country_norm"]
]

s2_lookup = s2.set_index("entity_id")[
    ["name_norm", "country_norm"]
]

s3_lookup = s3.set_index("entity_id")[
    ["name_norm", "country_norm"]
]

print("Fast recall test ready!")
print("Sample size:", len(sample_gt))

Fast recall test ready!
Sample size: 5000


In [19]:
# ==========================================
# STEP 6A: CREATE GROUND TRUTH SAMPLE
# ==========================================

sample_gt = gt.sample(
    n=min(50000, len(gt)),
    random_state=42
).copy()

print("Sample created successfully!")
print("Sample size:", len(sample_gt))

Sample created successfully!
Sample size: 50000


In [22]:
# ==========================================
# STEP 6B: FAST TOKEN BLOCKING RECALL
# ==========================================

covered = 0
total_matches = 0

for _, row in sample_gt.iterrows():

    s1_id = row["source1_entity_id"]

    # Source 1 information
    s1_name = s1_lookup.loc[s1_id, "name_norm"]
    s1_country = s1_lookup.loc[s1_id, "country_norm"]

    s1_tokens = set(get_name_tokens(s1_name))

    # Actual ground-truth matches
    matched_ids = str(row["matched_entity_ids"])

    if matched_ids.strip() == "" or matched_ids == "nan":
        continue

    actual_ids = [
        x.strip()
        for x in matched_ids.split(",")
        if x.strip()
    ]

    for entity_id in actual_ids:

        # Determine source from entity ID
        if entity_id.startswith("S2-"):
            lookup = s2_lookup
        elif entity_id.startswith("S3-"):
            lookup = s3_lookup
        else:
            continue

        # Get actual matched record
        try:
            match_name = lookup.loc[
                entity_id,
                "name_norm"
            ]

            match_country = lookup.loc[
                entity_id,
                "country_norm"
            ]
        except KeyError:
            continue

        total_matches += 1

        # Country must agree for our blocking strategy
        if s1_country != match_country:
            continue

        match_tokens = set(
            get_name_tokens(match_name)
        )

        # If at least one token is shared,
        # the blocking strategy would retrieve it.
        if s1_tokens.intersection(match_tokens):

            covered += 1


print("==========================================")
print("FAST TOKEN BLOCKING RESULTS")
print("==========================================")

print("Actual matches checked:", total_matches)
print("Matches covered:", covered)

if total_matches > 0:

    recall = covered / total_matches

    print(
        "Token blocking recall:",
        round(recall, 4)
    )

    print(
        "Token blocking recall (%):",
        round(recall * 100, 2),
        "%"
    )

FAST TOKEN BLOCKING RESULTS
Actual matches checked: 17216
Matches covered: 14771
Token blocking recall: 0.858
Token blocking recall (%): 85.8 %


In [23]:
# ==========================================
# STEP 7A: ADDRESS TOKENIZATION
# ==========================================

def get_address_tokens(address):
    if not address:
        return []
    
    tokens = address.split()
    
    # Ignore very short tokens
    return [
        token
        for token in tokens
        if len(token) >= 3
    ]


print("Address token function created successfully.")

Address token function created successfully.


In [24]:
# ==========================================
# STEP 7B: BUILD ADDRESS TOKEN INDEXES
# ==========================================

from collections import defaultdict

def build_address_index(df):
    index = defaultdict(list)

    for entity_id, address, country in zip(
        df["entity_id"],
        df["address_norm"],
        df["country_norm"]
    ):

        tokens = get_address_tokens(address)

        # Avoid duplicate tokens within the same address
        for token in set(tokens):
            index[(country, token)].append(entity_id)

    return index


print("Building Source 2 address index...")
s2_address_index = build_address_index(s2)

print("Building Source 3 address index...")
s3_address_index = build_address_index(s3)

print("\nAddress indexes created successfully!")

print("S2 address blocks:", len(s2_address_index))
print("S3 address blocks:", len(s3_address_index))

Building Source 2 address index...
Building Source 3 address index...

Address indexes created successfully!
S2 address blocks: 639019
S3 address blocks: 611431


In [26]:
# ==========================================
# STEP 7C: UPDATE LOOKUP TABLES
# ==========================================

s1_lookup = s1.set_index("entity_id")[
    ["name_norm", "address_norm", "country_norm"]
]

s2_lookup = s2.set_index("entity_id")[
    ["name_norm", "address_norm", "country_norm"]
]

s3_lookup = s3.set_index("entity_id")[
    ["name_norm", "address_norm", "country_norm"]
]

print("Lookup tables updated successfully!")
print("S1 columns:", s1_lookup.columns.tolist())
print("S2 columns:", s2_lookup.columns.tolist())
print("S3 columns:", s3_lookup.columns.tolist())

Lookup tables updated successfully!
S1 columns: ['name_norm', 'address_norm', 'country_norm']
S2 columns: ['name_norm', 'address_norm', 'country_norm']
S3 columns: ['name_norm', 'address_norm', 'country_norm']


In [27]:
# ==========================================
# STEP 7C: FAST ADDRESS BLOCKING RECALL
# ==========================================

covered = 0
total_matches = 0

for _, row in sample_gt.iterrows():

    s1_id = row["source1_entity_id"]

    # Source 1 address and country
    address = s1_lookup.loc[
        s1_id,
        "address_norm"
    ]

    country = s1_lookup.loc[
        s1_id,
        "country_norm"
    ]

    address_tokens = set(
        get_address_tokens(address)
    )

    # Actual ground-truth matches
    matched_ids = str(
        row["matched_entity_ids"]
    )

    if matched_ids.strip() == "" or matched_ids == "nan":
        continue

    actual_ids = [
        x.strip()
        for x in matched_ids.split(",")
        if x.strip()
    ]

    for entity_id in actual_ids:

        # Select correct source
        if entity_id.startswith("S2-"):
            lookup = s2_lookup

        elif entity_id.startswith("S3-"):
            lookup = s3_lookup

        else:
            continue

        try:
            match_address = lookup.loc[
                entity_id,
                "address_norm"
            ]

            match_country = lookup.loc[
                entity_id,
                "country_norm"
            ]

        except KeyError:
            continue

        total_matches += 1

        # Country must match
        if country != match_country:
            continue

        match_tokens = set(
            get_address_tokens(match_address)
        )

        # At least one shared address token
        if address_tokens.intersection(match_tokens):

            covered += 1


print("==========================================")
print("FAST ADDRESS BLOCKING RESULTS")
print("==========================================")

print(
    "Actual matches checked:",
    total_matches
)

print(
    "Matches covered:",
    covered
)

if total_matches > 0:

    recall = covered / total_matches

    print(
        "Address blocking recall:",
        round(recall, 4)
    )

    print(
        "Address blocking recall (%):",
        round(recall * 100, 2),
        "%"
    )

FAST ADDRESS BLOCKING RESULTS
Actual matches checked: 17216
Matches covered: 16438
Address blocking recall: 0.9548
Address blocking recall (%): 95.48 %


In [28]:
# ==========================================
# STEP 8: COMBINED BLOCKING RECALL
# ==========================================

covered = 0
total_matches = 0

for _, row in sample_gt.iterrows():

    s1_id = row["source1_entity_id"]

    name = s1_lookup.loc[
        s1_id,
        "name_norm"
    ]

    address = s1_lookup.loc[
        s1_id,
        "address_norm"
    ]

    country = s1_lookup.loc[
        s1_id,
        "country_norm"
    ]

    name_tokens = set(
        get_name_tokens(name)
    )

    address_tokens = set(
        get_address_tokens(address)
    )

    matched_ids = str(
        row["matched_entity_ids"]
    )

    if matched_ids.strip() == "" or matched_ids == "nan":
        continue

    actual_ids = [
        x.strip()
        for x in matched_ids.split(",")
        if x.strip()
    ]

    for entity_id in actual_ids:

        if entity_id.startswith("S2-"):
            lookup = s2_lookup

        elif entity_id.startswith("S3-"):
            lookup = s3_lookup

        else:
            continue

        try:
            match_name = lookup.loc[
                entity_id,
                "name_norm"
            ]

            match_address = lookup.loc[
                entity_id,
                "address_norm"
            ]

            match_country = lookup.loc[
                entity_id,
                "country_norm"
            ]

        except KeyError:
            continue

        total_matches += 1

        # Country must match
        if country != match_country:
            continue

        match_name_tokens = set(
            get_name_tokens(match_name)
        )

        match_address_tokens = set(
            get_address_tokens(match_address)
        )

        name_match = bool(
            name_tokens.intersection(
                match_name_tokens
            )
        )

        address_match = bool(
            address_tokens.intersection(
                match_address_tokens
            )
        )

        # Candidate survives if either
        # name OR address blocking succeeds
        if name_match or address_match:
            covered += 1


print("==========================================")
print("COMBINED BLOCKING RESULTS")
print("==========================================")

print(
    "Actual matches checked:",
    total_matches
)

print(
    "Matches covered:",
    covered
)

if total_matches > 0:

    recall = covered / total_matches

    print(
        "Combined blocking recall:",
        round(recall, 4)
    )

    print(
        "Combined blocking recall (%):",
        round(recall * 100, 2),
        "%"
    )

COMBINED BLOCKING RESULTS
Actual matches checked: 17216
Matches covered: 17206
Combined blocking recall: 0.9994
Combined blocking recall (%): 99.94 %
